In [2]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

### Compute day/night ratios to diasaggregate daily data to hourly data
To enable the model to run internally at hourly steps when daily forcings are provided.  
We use simple day/night ratios. In this method, 12 monthly night fractions are provided (one per month)  
Days are fixed as 06:00 t0 17:59. To get these ratios we use ERA Land hourly data

compute monthly climatological ratio  

$f_{night}$ = $\frac {\sum {P(night hours in month, m)}}{\sum {P(all hours in month, m)}}$


$P_{night} = f_{night} \cdot P_{d}$  
$f_{day} = 1 - f_{night}$  

For 12 day-hours:  
$P_{h} = P_{day}/12$  

For 12 night-hours:  
$P_{h} = P_{night}/12$  

In [ ]:
#Read ERA5-Land data Hourly Precipitation
#the file was deleted so merge the chunks again before running the script
data=xr.open_dataset('E:\VUB\_data\era5\era5land_tp_hourly_chunks/era5_hourly_P.nc')

In [3]:
#Extract Region of Interest data (Boechout)
data_ROI = data.sel(latitude=slice(51.48,51.048), longitude=slice(4.32,4.851)).mean(dim=['latitude','longitude'])

#cONVERT TO DATAFRAME
df_ROI=data_ROI.to_dataframe()[['tp']]

#Calculate day and night ratios per month to get 12 values
df_ROI['hour']=df_ROI.index.hour


d_n_ratios={}
for month in range(1,13):
    df_month=df_ROI[df_ROI.index.month==month]
    #calculate day and night sums
    day_sum=df_month[(df_month['hour']>=6) & (df_month['hour']<18)]['tp'].sum()
    night_sum=df_month[(df_month['hour']<6) | (df_month['hour']>=18)]['tp'].sum()
    day_ratio=day_sum/(day_sum+night_sum)
    night_ratio=night_sum/(day_sum+night_sum)
    d_n_ratios[month]={'day_ratio':day_ratio,'night_ratio':night_ratio}

#Convert to DataFrame for easier plotting
dn_ratios_df=pd.DataFrame.from_dict(d_n_ratios,orient='index')
dn_ratios_df.index.name='month'

#hourly ratios for plotting
hourly_day_ratios=[]
hourly_night_ratios=[]
for month in range(1,13):
    day_ratio=dn_ratios_df.loc[month,'day_ratio']
    night_ratio=dn_ratios_df.loc[month,'night_ratio']
    for hour in range(24):
        if 6 <= hour < 18:
            hourly_day_ratios.append(day_ratio/12)  # 12 daytime hours
        else:
            hourly_night_ratios.append(night_ratio/12)  # 12 nighttime hours

### Apply to daily data

In [4]:
#function to disaggregate daily precipitation into hourly values based on day/night ratios
def disaggregate_daily_to_hourly(daily_precip, dn_ratios_df, month):

    """
    Docstring to disaggregate_daily precipitation into hourly values based on day/night ratios.

    Parameters
    ----------
    daily_precip : float
        Daily precipitation amount.
    dn_ratios_df : pd.DataFrame
        DataFrame containing day and night ratios for each month.
    month : int
        Month for which to disaggregate (1-12).
    Returns
    -------
    hourly_precip : list
        List of 24 hourly precipitation values.
    """
    day_ratio=dn_ratios_df.loc[month,'day_ratio']
    night_ratio=dn_ratios_df.loc[month,'night_ratio']
    hourly_precip=[]
    for hour in range(24):
        if 6 <= hour < 18:
            hourly_precip.append(daily_precip * (day_ratio / 12))  # 12 daytime hours
        else:
            hourly_precip.append(daily_precip * (night_ratio / 12))  # 12 nighttime hours
    return hourly_precip

In [5]:
dn_ratios_df.to_csv('precipitation_day_night_ratios.csv')

In [ ]:
disaggregate_daily_to_hourly(10, dn_ratios_df, 1)  # Example usage